In [0]:
#MODULE 1 — utils/cdc_validator.py

# ==========================================================================
# MODULE   : utilities/cdc_validator.py
# PURPOSE  : Validate Change Data Feed (CDF) is enabled on Delta tables
# USAGE    : %run ./utilities/cdc_validator.py
#            result = validate_cdc(layer="gold", table_names=[...], catalog="hgi", schema="gold")
#
# FUNCTIONS:
#   validate_cdc()           — check CDF, return dict, continue pipeline
#   validate_cdc_and_raise() — check CDF, raise Exception if disabled (STOPS pipeline)
#   enable_cdc()             — enable CDF on a list of tables in one call
#
# WRITES TO:
#   {catalog}.logs.cdc_validation_log — one row per table per run
# ==========================================================================

from pyspark.sql import SparkSession

def validate_cdc(layer, table_names, catalog, schema):
    """
    Validates if CDF is enabled on specified Delta tables.
    Writes results to {catalog}.logs.cdc_validation_log.

    Args:
        layer       : str  — bronze, silver, or gold
        table_names : list — list of table names to check
        catalog     : str  — catalog name (e.g. hgi, bronze)
        schema      : str  — schema name (e.g. silver, gold, cust_001)

    Returns:
        dict — status, enabled_tables, disabled_tables, message
    """
    spark = SparkSession.getActiveSession()

    enabled_tables  = []
    disabled_tables = []
    error_tables    = []

    # Create CDC validation log table if not exists
    try:
        spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.logs")
        spark.sql(f"""
            CREATE TABLE IF NOT EXISTS {catalog}.logs.cdc_validation_log (
                job_name             STRING,
                customer_id          STRING,
                layer                STRING,
                table_name           STRING,
                cdf_status           STRING,
                validation_timestamp TIMESTAMP,
                run_date             DATE
            ) USING DELTA
        """)
    except Exception as e:
        print(f"  WARNING: Could not create cdc_validation_log: {str(e)}")

    print(f"\n  CDC Validation — Layer: {layer.upper()} | Catalog: {catalog} | Schema: {schema}")
    print("  " + "-"*60)

    for table_name in table_names:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        try:
            props = spark.sql(f"SHOW TBLPROPERTIES {full_table_name}").collect()
            cdf_enabled = False
            for row in props:
                if row["key"] == "delta.enableChangeDataFeed" and row["value"].lower() == "true":
                    cdf_enabled = True
                    break

            if cdf_enabled:
                enabled_tables.append(table_name)
                cdf_status = "ENABLED"
                print(f"  ✅ {full_table_name} — CDF ENABLED")
            else:
                disabled_tables.append(table_name)
                cdf_status = "DISABLED"
                print(f"  ❌ {full_table_name} — CDF DISABLED")

            # Write to cdc_validation_log
            try:
                spark.sql(f"""
                    INSERT INTO {catalog}.logs.cdc_validation_log VALUES (
                        'augment_job',
                        'cust_001',
                        '{layer}',
                        '{table_name}',
                        '{cdf_status}',
                        current_timestamp(),
                        current_date()
                    )
                """)
            except Exception as insert_e:
                print(f"  WARNING: Could not write to cdc_validation_log: {str(insert_e)}")

        except Exception as e:
            error_tables.append(table_name)
            print(f"  ⚠️  {full_table_name} — ERROR: {str(e)}")

    print("  " + "-"*60)

    all_enabled = len(disabled_tables) == 0 and len(error_tables) == 0

    if all_enabled:
        message = f"CDF enabled on all {len(enabled_tables)} tables in {layer} layer."
    else:
        message = f"CDF validation FAILED. Disabled: {disabled_tables}. Errors: {error_tables}."

    print(f"  RESULT: {'✅' if all_enabled else '❌'} {message}")

    return {
        "status":          all_enabled,
        "layer":           layer,
        "enabled_tables":  enabled_tables,
        "disabled_tables": disabled_tables,
        "error_tables":    error_tables,
        "message":         message
    }


def validate_cdc_and_raise(layer, table_names, catalog, schema):
    """
    Same as validate_cdc but raises Exception if any table has CDF disabled.
    Use this when you want the pipeline to STOP if CDF is not enabled.
    """
    result = validate_cdc(layer, table_names, catalog, schema)
    if not result["status"]:
        raise Exception(
            f"CDC Validation Failed — {result['message']} "
            f"Fix with: ALTER TABLE <table> SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')"
        )
    return result


def enable_cdc(table_names, catalog, schema):
    """
    Enables CDF on a list of tables in one call.
    Use this to fix disabled tables quickly.
    """
    spark = SparkSession.getActiveSession()
    print(f"\n  Enabling CDF on {len(table_names)} tables in {catalog}.{schema}...")
    for table_name in table_names:
        full_table_name = f"{catalog}.{schema}.{table_name}"
        try:
            spark.sql(f"""
                ALTER TABLE {full_table_name}
                SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')
            """)
            print(f"  ✅ CDF enabled on {full_table_name}")
        except Exception as e:
            print(f"  ❌ Failed on {full_table_name}: {str(e)}")


print("✅ cdc_validator.py loaded — functions: validate_cdc(), validate_cdc_and_raise(), enable_cdc()")
